In [165]:
import pandas as pd
import numpy as np

from src.paths import *

### Define a dictionary
fire me for these naming conventions

In [166]:
datasets = {
    'oc1' : 'OCS_1.csv',
    'oc2' : 'OCS_2.csv',
    'oc3' : 'OCS_3.csv',
    'oc4' : 'OCS_4.csv',
    'oc3_25' : 'OCS_3-25.csv',
    'oc3_5' : 'OCS_3-5.csv'
}

### Read CSV files

In [167]:
ds_df = {
    df : pd.read_csv(OUTPUT_PATH / filename, index_col=0)
    for df, filename in datasets.items()
}

C:\Users\Ehuan\AppData\Local\Temp\ipykernel_14556\2717829493.py:2: DtypeWarning: Columns (0: Parameter_Group) have mixed types. Specify dtype option on import or set low_memory=False.
  df : pd.read_csv(OUTPUT_PATH / filename, index_col=0)


### Final columns
the order of the outcome corresponds to this list

In [168]:
final_cols = [
    'date', 
    'fish_id', 
    'name', 
    'length', 
    'weight', 
    'gender', 
    'maturity', 
    'count', 
    'waterbody', 
    'lat', 
    'long', 
    'mp_name', 
    'p_name', 
    'p_group', 
    'concentration', 'p_flag', 'unit', 'rep_no',
    'detect_limit', 'mdl', 'is_surrogate'
    ]

### Drop the fork length

In [169]:
ds_df['oc3_25'] = ds_df['oc3_25'].drop(columns='Fork length (cm)')  

### Combine datasets

In [170]:
df = pd.concat([ds_df['oc1'], ds_df['oc2'], ds_df['oc3'], ds_df['oc3_25'], ds_df['oc3_5'], ds_df['oc4']], ignore_index=True)

### Delete unnecessary columns

In [171]:
df = df.drop(columns=[
    'Genus',  # Too Broad- can be used for different analyses
    'Species', # Too specific
    'Station_Name', # Need not know where it was tested
    'Basin/Site', # Redundant with the waterbody parameter
    'Measurement_Date', # Need not know when the fish length was measured
    'Collection_Year', # Redundant with date
    'Province/State', # Redundant with the waterbody parameter
    'Laboratory', # Don't need to know where it was tested
    'Method_Code'
])

### Rename columns

In [172]:
df = df.rename(columns={
    'CSP_No' : 'fish_id',
    'Common_Name' : 'name', 
    'Total Length (cm)' : 'length', 
    'Total Weight (g)': 'weight',
    'Sex_Code' : 'gender',
    'Tissue_Code' : 'tissue',
    'Maturity_Code' : 'maturity',
    'Composite_Count' : 'count',
    'Collection_Date' : 'date',
    'Latitude' : 'lat',
    'Longitude' : 'long',
    'Value' : 'concentration',
    'Value_Flag' : 'p_flag',
    'Master_Parameter_Name' : 'mp_name',
    'Parameter_name' : 'p_name',
    'Parameter_Group' : 'toxin_family',
    'Parameter_Code' : 'p_code',
    'Rep' : 'rep_no',
    'Unit_Code' : 'unit',
    'LOD' : 'detect_limit',
    'Waterbody' : 'waterbody',
    'MDL - typical' : 'mdl' # method detection limit
})

### See master parameter name and parameter name are different

In [173]:
# mask = ~(df["mp_name"].fillna("__NA__") == df["p_name"].fillna("__NA__"))

# df.loc[mask,["mp_name", "p_name"]].drop_duplicates()

### Determine surrogacy
some parameters contains lab-controlled chemicals

In [174]:
df['is_surrogate'] = df['mp_name'].str.contains('surrogate', case=False, na=False)

### Investigate any replicated observations

In [175]:
# print(df['CSP_No'].value_counts())

# print(df[df["fish_id"] == 351][["fish_id", "p_code", "mp_name", "p_name", "rep_no", "p_conc"]])

### Take the mean calculation of the replicated fishes
then drop the duplicates

In [176]:
df['concentration'] = df.groupby(['fish_id', 'p_code'])['concentration'].transform('mean')
df = df.drop_duplicates(subset=["fish_id", "p_code"])

### Determine if all fish got the same number of testing

In [177]:
rows_per_fish = df.groupby("fish_id").size()

# print(rows_per_fish.value_counts())
print("All same number of rows:", rows_per_fish.nunique() == 1)

All same number of rows: False


In [178]:
print(df['maturity'].isna().sum())
print(df[df['maturity'].isna()])


33547
        fish_id     name  length  weight gender maturity  count        date  \
0          7542  Alewife    12.0    15.6    NaN      NaN    5.0  1986-09-24   
1          7543  Alewife    14.6    28.2    NaN      NaN    5.0  1986-09-24   
2          7544  Alewife    15.3    33.0    NaN      NaN    5.0  1986-09-24   
3          7545  Alewife    16.1    34.6    NaN      NaN    5.0  1986-09-24   
4          7546  Alewife    16.4    35.7    NaN      NaN    5.0  1986-09-24   
...         ...      ...     ...     ...    ...      ...    ...         ...   
197605    67555  Walleye    41.5   660.2      F      NaN    1.0  2024-05-14   
197606    67555  Walleye    41.5   660.2      F      NaN    1.0  2024-05-14   
197607    67555  Walleye    41.5   660.2      F      NaN    1.0  2024-05-14   
197608    67555  Walleye    41.5   660.2      F      NaN    1.0  2024-05-14   
197609    67555  Walleye    41.5   660.2      F      NaN    1.0  2024-05-14   

           waterbody        lat  ...  concent

### Determine all the chemicals/toxins being tested for

In [179]:
df['mp_name'].value_counts().to_csv('mp_names.csv')

### Determine all tissue types

In [180]:
print(df['tissue'].value_counts())

tissue
WHA     191671
FIL       2580
RWHA        18
Name: count, dtype: int64


### Only keep rows where the whole fish is examined

In [181]:
df = df[df['tissue'] == 'WHA']
df = df.drop(columns=['tissue'])

### mp_name and p_name

determine if mpname and pname are different

In [188]:
df[['mp_name','p_name']].to_csv('name_list.csv')

new_df = df.copy()

df['p_name'] = df['p_name'].map({
  "'p-p'-DDE'":'ppDDE',

})

new_df['p_name'] = new_df['p_name'].str.lower()
new_df['mp_name'] = new_df['mp_name'].str.lower()


new_df['diff'] = df['mp_name'] != df['p_name']

print(new_df['diff'].value_counts())

new_df.to_csv('new.csv')


diff
True    191671
Name: count, dtype: int64


# Prepare for output

### Organize parameters

In [183]:
df = df[final_cols]

df = df.sort_values(by=['date','fish_id'])

KeyError: "['p_group'] not in index"

### Save cleaned dataframe to CSV

In [ ]:
cleaned = {
    'master_dataset.csv' : df,
}

for filename, dataframe in cleaned.items():
    dataframe.to_csv(CLEANED_PATH / filename, index=False)   